In [1]:
import ee
ee.Initialize(project="sentinel-487715")

# Ghana boundary
ghana = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
    .filter(ee.Filter.eq("country_na", "Ghana")) \
    .geometry()

# Roads asset
roads = ee.FeatureCollection("projects/sentinel-487715/assets/ghana_roads")

# Quarter ranges
quarters = {
    "Q1": ("01-01","03-31"),
    "Q2": ("04-01","06-30"),
    "Q3": ("07-01","09-30"),
    "Q4": ("10-01","12-31"),
}

def make_img(start, end):
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(ghana)
          .filterDate(start, end)
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
          .select(["B2","B3","B4","B8","B11","B12"])
          .median())

    ndvi = s2.normalizedDifference(["B8","B4"]).rename("NDVI")
    ndmi = s2.normalizedDifference(["B8","B11"]).rename("NDMI")
    ndbi = s2.normalizedDifference(["B11","B8"]).rename("NDBI")
    ndwi = s2.normalizedDifference(["B3","B8"]).rename("NDWI")
    bsi = (s2.select("B11").add(s2.select("B4"))
           .subtract(s2.select("B8").add(s2.select("B2")))
           .divide(s2.select("B11").add(s2.select("B4"))
                   .add(s2.select("B8")).add(s2.select("B2")))
           .rename("BSI"))

    return s2.addBands([ndvi, ndmi, ndbi, ndwi, bsi])

# Launch exports
tasks = []
for year in range(2020, 2024):
    for q, (start_mmdd, end_mmdd) in quarters.items():
        start = f"{year}-{start_mmdd}"
        end = f"{year}-{end_mmdd}"

        img = make_img(start, end)
        reducer = ee.Reducer.mean().forEachBand(img)

        stats_fc = img.reduceRegions(
            collection=roads,
            reducer=reducer,
            scale=20,
            tileScale=16
        ).map(lambda f: f.set({"year": year, "quarter": q}))

        desc = f"ghana_roads_{year}_{q}"
        task = ee.batch.Export.table.toDrive(
            collection=stats_fc,
            description=desc,
            fileFormat="CSV",
            folder="GEE_Exports"
        )
        task.start()
        tasks.append(task)

print(f"Started {len(tasks)} export tasks.")


Started 16 export tasks.
